# Tool Calling / Function Calling

**Week 2 Day 4 - Learning Lab**

Implementing function calling where LLMs can call Python functions as tools.

## Intent

Learn to:
- Define tools using JSON schema
- Implement function registry pattern (no if/elif chains)
- Handle tool calls manually (understand the mechanics)
- Store conversation history in SQLite (persistent storage)
- Combine streaming with tool calls (hybrid approach)
- Implement session management for multi-user conversations
- Handle errors gracefully in tool execution

## Expected Insights

- Function registry pattern scales beautifully (2 tools or 200 tools)
- SQLite for conversation history > in-memory storage for production
- Manual implementation teaches you what SDKs abstract away
- Streaming + tool calls requires hybrid approach (stream → detect → non-stream → execute → stream)
- Comprehensive error handling essential for production apps
- Pattern: Tools enable LLMs to interact with databases, APIs, and external systems


In [ ]:
# Setup
import os
import json
import sqlite3
import uuid
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
from typing import Dict, Callable, List, Any, Optional

load_dotenv(override=True)
client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))


## Experiment 1: Tool Definition and Basic Tool Calling

**Key Pattern:** Define tool using JSON schema, pass to API, handle tool calls.

**Why:** Understand the fundamental mechanics of tool calling.

**Future me:** This is the foundation - everything else builds on this pattern.


In [ ]:
# Example: Simple tool function
def get_weather(city: str) -> str:
    """Get weather for a city (mock implementation)."""
    return f"The weather in {city} is sunny, 72°F"

# Tool definition (JSON schema)
# This schema tells the LLM what the function does and what parameters it needs
weather_tool = {
    "name": "get_weather",
    "description": "Get the current weather for a city",
    "parameters": {
        "type": "object",
        "properties": {
            "city": {
                "type": "string",
                "description": "The city to get weather for"
            }
        },
        "required": ["city"],
        "additionalProperties": False
    }
}

tools = [{"type": "function", "function": weather_tool}]

# Basic tool calling flow
# This shows the complete flow - API call → tool detection → execution → continuation
messages = [
    {"role": "user", "content": "What's the weather in Paris?"}
]

response = client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=messages,
    tools=tools
)

# Check if tool was called
if response.choices[0].finish_reason == "tool_calls":
    tool_call = response.choices[0].message.tool_calls[0]
    function_name = tool_call.function.name
    arguments = json.loads(tool_call.function.arguments)
    
    # Execute tool
    result = get_weather(**arguments)
    
    # Add tool response (must include tool_call_id!)
    messages.append(response.choices[0].message)
    messages.append({
        "role": "tool",
        "content": result,
        "tool_call_id": tool_call.id
    })
    
    # Continue conversation
    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=messages
    )
    
    print(response.choices[0].message.content)


## Experiment 2: Function Registry Pattern

**Key Pattern:** Dictionary-based registry eliminates if/elif chains.

**Why:** Scales beautifully - adding new tool = one line in registry.

**Future me:** This is the key improvement - no more if/elif chains that don't scale!


In [ ]:
# Function registry (no if/elif chains!)
# This pattern scales from 2 tools to 200 tools without code changes
TOOL_REGISTRY: Dict[str, Callable] = {
    "get_weather": get_weather,
    # Add more tools here - just one line per tool
    # "get_ticket_price": get_ticket_price,
    # "set_ticket_price": set_ticket_price,
}

def handle_tool_calls(message) -> List[Dict[str, Any]]:
    """Handle tool calls using registry pattern.
    
    Future me: This is where the magic happens - dictionary lookup replaces
    if/elif chains. To add a new tool, just add it to TOOL_REGISTRY above.
    """
    responses = []
    
    for tool_call in message.tool_calls:
        function_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        
        # Look up function in registry
        if function_name in TOOL_REGISTRY:
            func = TOOL_REGISTRY[function_name]
            result = func(**arguments)  # Unpack dict as keyword arguments
            
            responses.append({
                "role": "tool",
                "content": result,
                "tool_call_id": tool_call.id
            })
    
    return responses


## Experiment 3: SQLite for Conversation History

**Key Pattern:** Store conversation history in SQLite instead of in-memory.

**Why:** Persistent, queryable, production-ready storage.

**Future me:** Gradio's in-memory history is ephemeral - SQLite is the way to go for real apps.


In [ ]:
# Database setup
# This schema stores all conversation history with tool calls
DB_CONVERSATIONS = "conversations.db"

with sqlite3.connect(DB_CONVERSATIONS) as conn:
    cursor = conn.cursor()
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS conversations (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            session_id TEXT NOT NULL,
            role TEXT NOT NULL,
            content TEXT,
            tool_calls TEXT,  # JSON for assistant messages, tool_call_id for tool messages
            timestamp DATETIME DEFAULT CURRENT_TIMESTAMP
        )
    ''')
    cursor.execute('''
        CREATE INDEX IF NOT EXISTS idx_session ON conversations(session_id)
    ''')
    conn.commit()

# Save message
# Every message (system, user, assistant, tool) gets saved here
def save_message(session_id: str, role: str, content: str, tool_calls: Optional[str] = None):
    with sqlite3.connect(DB_CONVERSATIONS) as conn:
        cursor = conn.cursor()
        cursor.execute(
            'INSERT INTO conversations (session_id, role, content, tool_calls) VALUES (?, ?, ?, ?)',
            (session_id, role, content, tool_calls)
        )
        conn.commit()

# Load conversation
# Reconstruct full conversation from database, including tool calls
def load_conversation(session_id: str) -> List[Dict[str, Any]]:
    with sqlite3.connect(DB_CONVERSATIONS) as conn:
        cursor = conn.cursor()
        cursor.execute(
            'SELECT role, content, tool_calls FROM conversations WHERE session_id = ? ORDER BY id',
            (session_id,)
        )
        rows = cursor.fetchall()
        
        messages = []
        for role, content, tool_calls_json in rows:
            msg = {"role": role, "content": content or ""}
            
            # Reconstruct assistant messages with tool_calls
            if role == "assistant" and tool_calls_json:
                msg["tool_calls"] = json.loads(tool_calls_json)
            
            # Reconstruct tool messages with tool_call_id
            elif role == "tool" and tool_calls_json:
                msg["tool_call_id"] = tool_calls_json  # tool_calls column stores tool_call_id for tool messages
            
            messages.append(msg)
        
        return messages


## Experiment 4: Error Handling for Tool Execution

**Key Pattern:** Comprehensive error handling at each failure point.

**Why:** Tools can fail in many ways - need graceful error handling.

**Future me:** Production apps must handle all edge cases gracefully.


In [ ]:
def handle_tool_calls_with_errors(message) -> List[Dict[str, Any]]:
    """Handle tool calls with comprehensive error handling.
    
    This shows all the error cases we need to handle:
    - JSON parsing errors (invalid JSON in arguments)
    - TypeError (wrong number/type of arguments)
    - General exceptions (runtime errors in tool functions)
    - Unknown tools (not found in registry)
    """
    responses = []
    
    for tool_call in message.tool_calls:
        function_name = tool_call.function.name
        
        try:
            arguments = json.loads(tool_call.function.arguments)
        except json.JSONDecodeError as e:
            responses.append({
                "role": "tool",
                "content": f"Error: Invalid tool arguments JSON. {str(e)}",
                "tool_call_id": tool_call.id
            })
            continue
        
        if function_name in TOOL_REGISTRY:
            func = TOOL_REGISTRY[function_name]
            try:
                result = func(**arguments)
                responses.append({
                    "role": "tool",
                    "content": result,
                    "tool_call_id": tool_call.id
                })
            except TypeError as e:
                # Wrong number of arguments or wrong argument names
                responses.append({
                    "role": "tool",
                    "content": f"Error: Function {function_name} received invalid arguments. {str(e)}",
                    "tool_call_id": tool_call.id
                })
            except Exception as e:
                # Any other error from the function itself
                responses.append({
                    "role": "tool",
                    "content": f"Error executing {function_name}: {str(e)}",
                    "tool_call_id": tool_call.id
                })
        else:
            responses.append({
                "role": "tool",
                "content": f"Error: Unknown tool '{function_name}'",
                "tool_call_id": tool_call.id
            })
    
    return responses


## Experiment 5: Streaming + Tool Calls (Hybrid Approach)

**Key Pattern:** Stream when possible, switch to non-streaming for tools.

**Why:** Streaming provides better UX, but tool calls require complete data.

**Future me:** This is complex but necessary - see day4.ipynb for complete implementation.


In [ ]:
# Simplified example of hybrid streaming
# The complete implementation is in week2/day4.ipynb
# This shows the key pattern: detect tools in stream → switch to non-streaming → execute → resume streaming

def chat_with_tools(message: str, session_id: str):
    """Chat function with streaming and tool calls (simplified example)."""
    messages = load_conversation(session_id)
    messages.append({"role": "user", "content": message})
    
    # Try streaming first
    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=messages,
        tools=tools,
        stream=True
    )
    
    # Detect tool calls in stream
    # Track finish_reason from stream chunks (not from Stream object)
    tool_calls_detected = False
    finish_reason = None
    accumulated_content = ""
    
    for chunk in response:
        if chunk.choices[0].delta.content:
            accumulated_content += chunk.choices[0].delta.content
            yield accumulated_content
        
        if chunk.choices[0].delta.tool_calls:
            tool_calls_detected = True
        
        if chunk.choices[0].finish_reason:
            finish_reason = chunk.choices[0].finish_reason
    
    # If tool calls detected, switch to non-streaming
    # Can't get complete tool call data from stream, need non-streaming call
    if tool_calls_detected or finish_reason == "tool_calls":
        response = client.chat.completions.create(
            model="gpt-4.1-mini",
            messages=messages,
            tools=tools
        )
        
        # Handle tool calls (see day4.ipynb for complete implementation)
        # ... tool execution logic ...
        
        # Resume streaming after tools executed
        # ... streaming logic ...


## Key Takeaways

### Function Registry Pattern
- Dictionary-based lookup eliminates if/elif chains
- Scales from 2 tools to 200 tools without code changes
- Self-documenting (registry shows all available tools)
- Industry-standard pattern for extensible systems

### SQLite for Conversation History
- Persistent storage (survives restarts)
- Queryable (can analyze conversations)
- Production-ready (real-world apps need database-backed history)
- Tool calls must be properly stored and reconstructed

### Manual Tool Calling Implementation
- Understand what happens under the hood
- Learn complete flow: tool detection → execution → response → continuation
- Practice handling edge cases (multiple tools, errors, streaming)
- Build foundation for understanding agent frameworks

### Streaming + Tool Calls
- Hybrid approach: stream when possible, switch to non-streaming for tools
- Track finish_reason from stream chunks (not from Stream object)
- Check response type before accessing attributes (Stream vs ChatCompletion)

### Error Handling
- Comprehensive error handling essential for production
- Handle JSON parsing errors, TypeError, general exceptions
- Return errors as tool responses so LLM can handle them

### Session Management
- Use session_id to separate conversations per user/browser
- Generate unique ID from Gradio request or UUID fallback
- Filter database queries by session_id for multi-user support

## Reference

For complete production-ready implementation, see `week2/day4.ipynb` (Airline Assistant).
